In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", dtype=torch.float16, device_map="auto", attn_implementation="sdpa")
# tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

# input_ids = tokenizer("Hello, I'm a language model", return_tensors="pt").to(model.device)

# output = model.generate(**input_ids, cache_implementation="static")
# print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 668.97it/s]
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
c:\Users\jarre\ics 435 machine learning fundamentals\assignments\assignment4\.env\Lib\site-packages\transformers\generation\utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=27) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Hello, I'm a language modeler. I'm a programmer. I'm a writer. I'm a writer. I'm a


In [29]:
from datasets import Dataset
import pandas as pd

df = Dataset.from_pandas(pd.read_csv('data.csv')).remove_columns('ID')
df = df.train_test_split(test_size=.2, shuffle=True
                        #  , seed=42
                        )
train = df['train']
test = df['test']

In [30]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling, Trainer, TrainingArguments

tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token 
model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8325.38it/s]


In [31]:
def tokenize(batch):
    return tokenizer(batch['Joke'], truncation=True, padding=True)

In [32]:
tokenized_train = train.map(tokenize, batched=True, remove_columns=['Joke'])
tokenized_test = test.map(tokenize, batched=True, remove_columns=['Joke'])

Map: 100%|██████████| 325/325 [00:00<00:00, 9340.92 examples/s]


In [33]:
block_size = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

blocked_tokenized_train = tokenized_train.map(group_texts, batched=True)
blocked_tokenized_test = tokenized_test.map(group_texts, batched=True)

Map: 100%|██████████| 325/325 [00:00<00:00, 7105.65 examples/s]


In [34]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [35]:
training_args = TrainingArguments(
    output_dir="./gpt2-jokes",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_strategy="epoch",
    logging_steps=10,
)

# 7. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=blocked_tokenized_train,
    eval_dataset=blocked_tokenized_test,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss
1,4.136414,4.027392
2,3.676627,3.849323
3,3.683530,3.823503


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]


TrainOutput(global_step=1020, training_loss=4.186994892943139, metrics={'train_runtime': 60.9783, 'train_samples_per_second': 33.405, 'train_steps_per_second': 16.727, 'total_flos': 133062967296000.0, 'train_loss': 4.186994892943139, 'epoch': 3.0})

In [36]:
input_ids = tokenizer("what did the", return_tensors="pt").to(model.device)

output = model.generate(**input_ids, cache_implementation="static")
print(tokenizer.decode(output[0], skip_special_tokens=True))

what did the cow say to the cow? "What do you call a cow with no legs?" "What do


c:\Users\jarre\ics 435 machine learning fundamentals\assignments\assignment4\.env\Lib\site-packages\transformers\generation\utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=23) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
